# SSM Centric Indexer
```
ssm{}
    |____ consequence[]
    |           |_____ transcript{}
    |                        |_____ gene{}
    |                        |_____ annotation{}
    |____ occurrence[]
                |_____ case{}
                         |____ observation[]
```

In [2]:
import os
import requests
import uuid
%load_ext autoreload
from exports.mappings import GeneMapper, SSMMapper, Mapper
from exports.utils import get_array_paths

from pyspark.sql.functions import col, explode, collect_list, size, sum, first, struct, udf, regexp_extract, lit, count, broadcast
from pyspark.sql.types import StringType

## Load combined maf into spark

In [3]:
url = 's3a://test/combined_mafs.csv'
    
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true')\
                .load(url)\
                .drop_duplicates()

In [4]:
#df = df.limit(500000)

In [5]:
'''
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true', comment='#', delimiter='\t')\
                .load('/home/ubuntu/tests/data/test.maf')
'''        

"\ndf = sqlContext.read.format('com.databricks.spark.csv')                .options(header='true', inferschema='true', comment='#', delimiter='\t')                .load('/home/ubuntu/tests/data/test.maf')\n"

## Rename and select desired columns in the mafs

In [6]:
%autoreload
from exports.utils import (
    maf_annotation_map,
    maf_gene_map,
    maf_observation_map,
    maf_ssm_map,
    maf_transcript_map,
    tumor_genotype_map,
    tumor_validation_map,
    normal_genotype_map,
    sample_map,
    input_bam_map,
    read_depth_map,
    maf_cols
)

maf_df = df.select(*( col(v).alias(k) for k,v in maf_cols.items() ))

## Augment maf df by extracting submitter_id and creating ssm_uuids

In [7]:
maf_df = maf_df.withColumn('_case_submitter_id',
                           regexp_extract(col('tumor_sample_barcode'),
                                          '([A-Z]{4}-[A-Z0-9]{2}-[A-Z0-9]{4})',1))
maf_ssm_map.update({'_case_submitter_id':'_case_submitter_id'})

In [8]:
def ssm_uuid(chromosome, start_position, ref_allele, tumor_allele):
    '''
    SNP: "{chromosome}:g.{start_position}{reference_allele}>{tumor_allele}"
    DEL: "{chromosome}:g.{start_position}del{reference_allele}"
    INS: "{chromosome}:g.{start_position}_{end_position}ins{tumor_allele}"
    '''
    chromosome = chromosome.replace('chr','')
    label = '{}:g.{}:{}>{}'.format(chromosome, start_position, ref_allele, tumor_allele)
    return str(uuid.uuid5(uuid.UUID('d15296a3-38ed-412e-8ace-75e235f82f55'), label))

ssm_uuid_udf =udf(ssm_uuid, StringType())
maf_df = maf_df.withColumn('ssm_uuid', ssm_uuid_udf(col('chromosome'), col('start_position'), col('reference_allele'), col('tumor_allele')))
maf_observation_map.update({'ssm_uuid':'ssm_uuid'})
maf_ssm_map.update({'ssm_uuid':'ssm_uuid'})

## Slice and dice until we get to the format we want

### SSM df

In [9]:
ssm_df = maf_df.select(*( col(k) for k in maf_ssm_map.keys() ))\
               .drop_duplicates()

## Transcript + Annotation + Gene
```
consequence[]
    |_____ transcript{}
                |_____ gene{}
                |_____ annotation{}
```

In [10]:
# Select annotation and gene into nested format
tran_df = maf_df.select('gene_symbol', struct(
                                struct(*maf_annotation_map.keys()).alias('annotation'),
                                struct(*maf_gene_map.keys()).alias('gene'),
                                *maf_transcript_map.keys()
                              ).alias('transcript'))\
                            .drop_duplicates()

In [11]:
consequence_df = tran_df.select('gene_symbol', struct('transcript').alias('consequence'))\
                        .groupBy('gene_symbol')\
                        .agg(collect_list('consequence').alias('consequence'))

## Observation + Case + Occurance
```
occurrence[]
    |_____ case{}
             |____ observation[]
```

### Observation df

In [12]:
observation_df = maf_df.select('_case_submitter_id',
                               *(maf_observation_map.keys()
                                 +normal_genotype_map.keys()
                                 +tumor_genotype_map.keys()
                                 +tumor_validation_map.keys()
                                 +read_depth_map.keys()
                                 +input_bam_map.keys()
                                 +sample_map.keys()))

observation_df = observation_df.select('_case_submitter_id',
                                       struct(*normal_genotype_map.keys()).alias('normal_genotype'),
                                       struct(*tumor_genotype_map.keys()).alias('tumor_genotype'),
                                       struct(*tumor_validation_map.keys()).alias('validation'),
                                       struct(*read_depth_map.keys()).alias('read_depth'),
                                       struct(*input_bam_map.keys()).alias('input_bam_file'),
                                       struct(*sample_map.keys()).alias('sample'),
                                       *maf_observation_map.keys())\
                                .drop('ssm_uuid')\
                                .drop_duplicates()

In [13]:
#observation_df.count()
#observation_df.printSchema()

### Get case dataframe from existing graph

In [14]:
#doc = requests.get('http://elasticsearch.service.consul:9200/gdc_from_graph_35/_search').json()['hits']['hits'][0]['_source']
case_df = sqlContext.read.format("es")\
    .option('es.nodes', 'elasticsearch.service.consul')\
    .option('es.read.field.include', 'case_id,submitter_id,state,project.*,program.*,exposures.*,demographic.*')\
    .option('es.read.field.as.array.include','')\
    .option('es.resource.read', 'gdc_from_graph/case')\
    .option('es.nodes.resolve.hostname','false')\
    .load("gdc_from_graph")

In [15]:
# Merge observation with Case
occurence_df = case_df.join(observation_df, case_df.submitter_id == observation_df._case_submitter_id, 'left')\
                        .select('submitter_id', struct(struct(struct(observation_df.columns).alias('observation'),*case_df.columns).alias('case')).alias('occurence'))\
                        .groupby('submitter_id')\
                        .agg(collect_list('occurence').alias('occurence'))

## Assemble constituent parts
```
ssm{}
    |____ consequence[]
    |           |_____ transcript{}
    |                        |_____ gene{}
    |                        |_____ annotation{}
    |____ occurrence[]
                |_____ case{}
                         |____ observation[]
```

In [16]:
ssm_df.repartition(8192,'gene_symbol').persist().count()

513688

In [17]:
ssm_centric = ssm_df.sample(False, 0.1).join(occurence_df, ssm_df._case_submitter_id == occurence_df.submitter_id, 'left')\
                    .join(consequence_df, ssm_df.gene_symbol == consequence_df.gene_symbol, 'left')

In [18]:
ssm_df.count()

513688

In [19]:
ssm_centric.count()

50771

## Export df to es

In [20]:
sqlContext.sql("set spark.sql.shuffle.partitions=4096")

DataFrame[key: string, value: string]

#### Graph es index

In [21]:
#print requests.get('http://elasticsearch.service.consul:9200/_cat/indices?v').text

#### New vis index

In [22]:
print requests.get('http://elasticsearchvis.service.consul:9200/_cat/indices?v').text

health status index                           uuid                   pri rep docs.count docs.deleted store.size pri.store.size
green  open   .monitoring-kibana-2-2016.11.21 Gg2lRUkbSpOJ69-so0TSpg   1   1       4520            0      2.2mb            1mb
green  open   .monitoring-kibana-2-2016.11.20 sYEaQ1ukStibxpHDnPI0Zg   1   1      17188            0      7.6mb          3.8mb
green  open   .monitoring-data-2              pt6so5oJRiCd9z0JwoJyKQ   1   1         10            0     39.2kb         19.6kb
green  open   case                            U_5pmk4dRpahgx7jXs2u5Q  10   0    4087345            0    608.9mb        608.9mb
green  open   .monitoring-es-2-2016.11.19     cpecQtt3Tx-Y1HP0BP0tpQ   1   1     253250          666    324.6mb          162mb
green  open   ssm                             vCF3QIyvT0m_ETj3T_CjFw   8   0          0            0        1kb            1kb
green  open   .monitoring-kibana-2-2016.11.18 8Cxzih0JQzSEeCFLLBUgaw   1   1      16392            0      7.4mb

In [23]:
print requests.get('http://elasticsearchvis.service.consul:9200/_cat/count/test/ssm?v').text

No handler found for uri [/_cat/count/test/ssm?v] and method [GET]


In [24]:
%autoreload
from exports.mappings import CaseMapper, SSMMapper
m = SSMMapper()

In [25]:
m.mapping['dynamic'] = 'true'
#m.mapping['properties']['ssm']['dynamic'] = 'true'
#m.mapping['properties']['gene']['properties']['ssm']['dynamic'] = 'true'
#m.mapping['properties']['gene']['properties']['ssm']['properties']['observation']['dynamic'] = 'true'
#m.mapping['properties']['case']['properties']['files']['properties']['cases']['dynamic'] = 'true'

In [27]:
import json

print requests.delete('http://elasticsearchvis.service.consul:9200/ssm').json()

data = json.dumps({"settings":{"index":{
                "refresh_interval":"1m",
                "number_of_shards":8,
                "number_of_replicas":0,
                "mapper.dynamic":False,
                "mapping.nested_fields.limit":100,
                "mapping.total_fields.limit":2000
            }},"mappings":{
                "ssm":m.mapping
            }})
#print requests.put('http://localhost:9200/test/', data=data).json()
print requests.put('http://elasticsearchvis.service.consul:9200/ssm', data=data).json()

{u'acknowledged': True}
{u'acknowledged': True, u'shards_acknowledged': True}


In [ ]:
#%%time
ssm_centric.write.format('org.elasticsearch.spark.sql')\
                    .option('es.nodes', 'elasticsearchvis.service.consul')\
                    .option('es.nodes.resolve.hostname','false')\
                    .option('es.resource.write', 'ssm/ssm')\
                    .option('es.http.timeout', '10m')\
                    .option('es.http.retries', '300')\
                    .option('es.batch.write.retry.count', '100')\
                    .option('es.batch.write.retry.wait', '10m')\
                    .option('es.batch.size.bytes','5mb')\
                    .option('es.batch.size.entries', '1000')\
                    .option('es.batch.write.refresh', 'true')\
                    .save('ssm/ssm')
            
#requests.post('http://localhost:9200/test-case/_refresh')